<a href="https://colab.research.google.com/github/Stav788/archaeo-ner-greek/blob/main/notebooks/gliner2_training_v12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning GLiNER2 for Archaeological Named Entity Recognition


### Colab Configuration Requirements
To run this notebook on Google Colab, add the following secrets to your environment (Key icon in the left sidebar):
#### 1. Repository Access
* **`GITHUB_TOKEN`**: Required for cloning private source code.
* **Obtain**: [GitHub Settings](https://github.com/settings/tokens) > Developer Settings > Personal access tokens. Required scope: `repo` or `contents:read`.
#### 2. Hugging Face Integration
* **`HF_TOKEN`**: Required for accessing the dataset.
* **Obtain**: [HF Settings](https://huggingface.co/settings/tokens).
#### 3. Activation
* Toggle **Notebook access** to **ON** for all listed secrets.

## Environment Initialization & Configuration

In [ ]:
import os
import sys
import subprocess

# --- STANDALONE BOOTSTRAP FOR COLAB ---
# Clones the repo and sets paths BEFORE internal package imports.
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("\n>>> Colab Pipeline Version: 1.5.2\n")
    # 1. Self-Healing check for corrupted PIL/Pillow
    try:
        from PIL import Image, ImageFont
        from PIL._typing import _Ink
    except ImportError:
        print("\n[Self-Healing] Mismatched/corrupted PIL installation detected. Force-reinstalling and restarting runtime...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--force-reinstall", "-q", "-U", "Pillow"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "protobuf", "torchao", "transformers", "huggingface_hub==0.25.2"])
        print("\n[Self-Healing] Restarting active Python kernel to load clean Pillow files. Please wait 3 seconds...")
        import os
        os.kill(os.getpid(), 9)

    # 2. Install critical requirements if not already done
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "protobuf", "torchao", "transformers", "huggingface_hub==0.25.2"])
    os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
    try:
        from google.colab import userdata
        def get_sec(k):
            val = userdata.get(k)
            return val.strip() if val else None

        GITHUB_TOKEN = get_sec('GITHUB_TOKEN')
        REPO_NAME = "archaeo-ner-greek"

        if GITHUB_TOKEN:
            masked = f"{GITHUB_TOKEN[:4]}...{GITHUB_TOKEN[-4:]}"
            print(f"GITHUB_TOKEN loaded (Length: {len(GITHUB_TOKEN)}, Masked: {masked})")
        else:
            print("GITHUB_TOKEN is MISSING in Colab Secrets.")

        if not os.path.exists(REPO_NAME):
            REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Stav788/{REPO_NAME}.git"
            try:
                result = subprocess.run(["git", "clone", "--branch", "dev", REPO_URL],
                                       capture_output=True, text=True)
                if result.returncode != 0:
                    clean_err = result.stderr.replace(GITHUB_TOKEN, "********") if GITHUB_TOKEN else result.stderr
                    print(f"Warning: Bootstrap clone failed.\nGit Error: {clean_err}")
            except Exception as e:
                print(f"Warning: Unexpected error during clone: {e}")
        else:
            print("Repository already exists. Pulling latest changes from dev branch...")
            try:
                result = subprocess.run(["git", "-C", REPO_NAME, "pull"], capture_output=True, text=True)
                if result.returncode != 0:
                    clean_err = result.stderr.replace(GITHUB_TOKEN, "********") if GITHUB_TOKEN else result.stderr
                    print(f"Warning: Bootstrap pull failed.\nGit Error: {clean_err}")
                else:
                    print("Successfully pulled latest changes!")
            except Exception as e:
                print(f"Warning: Unexpected error during pull: {e}")

        if os.path.exists(REPO_NAME):
            if os.path.abspath(REPO_NAME) not in sys.path:
                sys.path.append(os.path.abspath(REPO_NAME))
            os.chdir(REPO_NAME)
        else:
            print("Warning: Repository folder missing. Using pre-installed modules if available.")
    except Exception as e:
        print(f"Colab bootstrap failed: {e}")

In [ ]:
import json
import logging
import warnings
import random
from datetime import datetime
from datasets import load_dataset
from logging.config import dictConfig
from pathlib import Path

# Try to import wandb (optional)
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False

from archaeo_ner_greek.training_utils import (
    setup_local, setup_colab, df_to_gliner_examples,
    plot_training_history, plot_threshold_curves,
    plot_ner_confusion_matrix, compute_metrics, get_cnt, evaluate_adapter,
    show_error_analysis, show_detailed_report,
    safe_wandb_log, setup_wandb, upload_wandb_artifact
)

# 1. Environment Detection & Pre-Import Setup
if IN_COLAB:
    env_vars = setup_colab()
else:
    env_vars = setup_local()

# 2. Optimized Imports
from tabulate import tabulate
import matplotlib.pyplot as plt
import torch
from gliner2 import GLiNER2
from gliner2.training.data import InputExample, TrainingDataset
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Local project imports
import archaeo_ner_greek
from archaeo_ner_greek.logging_config import setup_logging

# 3. Path Management
BASE_DIR = Path(os.getcwd())
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = DATA_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 4. Logging & Global Config
log_file = setup_logging()
logger = logging.getLogger(__name__)
logger.info(f">>> Logging to: {log_file}")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow")
warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*SwigPyObject.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="wandb")

logger.info(f">>> Working Directory: {BASE_DIR}")
logger.info(f">>> Models Directory:  {MODELS_DIR}")
logger.info(f">>>> Dataset Repo:      {env_vars.get('HF_REPO_ID', 'Stalexan/archaeo-ner-greek')}")

# WandB Status for later logging
logger.info(">>> SECRETS DIAGNOSTIC")
MANDATORY_SECRETS = ["HF_TOKEN"]
OPTIONAL_SECRETS  = ["GITHUB_TOKEN", "WANDB_API_KEY"]

missing_mandatory = []
for key in MANDATORY_SECRETS:
    if not env_vars.get(key):
        logger.error(f"FATAL: Mandatory secret '{key}' is MISSING.")
        missing_mandatory.append(key)
    else:
        logger.info(f"Secret '{key}': FOUND")

for key in OPTIONAL_SECRETS:
    if not env_vars.get(key):
        logger.warning(f"ADVISORY: Optional secret '{key}' is MISSING. (Continuing...)")
    else:
        logger.info(f"Secret '{key}': FOUND")

if missing_mandatory:
    logger.error(f"Cannot continue. Missing mandatory secrets: {missing_mandatory}")
    raise ValueError(f"Missing mandatory secrets: {missing_mandatory}")

wandb_enabled = WANDB_AVAILABLE and env_vars.get("WANDB_API_KEY") is not None
if not wandb_enabled:
    logger.warning("WandB is DISABLED (Missing key). Continuing without cloud logging.")
else:
    logger.info("WandB is ENABLED.")

## Data Loading from Hugging Face

We fetch the `default` configuration partitions (Train/Val/Test) from the official HF repository.
These splits are already document-aware and stratified.

In [ ]:
repo_id = env_vars.get("HF_REPO_ID", "Stalexan/archaeo-ner-greek")
hf_token = env_vars.get("HF_TOKEN") or env_vars.get("HUGGING_FACE_HUB_TOKEN")

logger.info(f"Loading partitions from HF: {repo_id} (Subset: default)")
ds = load_dataset(repo_id, name="default", token=hf_token)

# Map splits to DataFrames for compatibility with downstream conversion
df_train = ds["train"].to_pandas()
df_val = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()

logger.info(f"HF Partitions Loaded: Train={len(df_train)}, Val={len(df_val)}, Test={len(df_test)}")
total_records = len(df_train) + len(df_val) + len(df_test)
logger.info(f"Total Records: {total_records}")
logger.info(f"Final Split Ratios: Train={len(df_train)/total_records:.1%}, Val={len(df_val)/total_records:.1%}, Test={len(df_test)/total_records:.1%}")

### Semantic Schema & Entity Label Definitions

In [ ]:
# Dynamically find the package resources folder
PACKAGE_ROOT = Path(archaeo_ner_greek.__file__).parent
RESOURCES_DIR = PACKAGE_ROOT / "resources"
GUIDELINES_PATH = RESOURCES_DIR / "archaeoner_labels_definitions_v12_st.json"
logger.info(f"Loading entity descriptions from {GUIDELINES_PATH}")
with open(GUIDELINES_PATH, 'r', encoding='utf-8') as f:
    entity_descriptions = json.load(f)

logger.info(f"Labels: {list(entity_descriptions.keys())}")
logger.info(f"Example: ARTEFACT: {entity_descriptions['ARTEFACT']}")

### Dataset Serialization for GLiNER2

In [ ]:
train_examples = df_to_gliner_examples(df_train, entity_descriptions)
val_examples   = df_to_gliner_examples(df_val, entity_descriptions)
test_examples  = df_to_gliner_examples(df_test, entity_descriptions)

logger.info(f"Text Sample: {train_examples[0].text}")
logger.info(f"Entities Sample: {train_examples[0].entities}")

In [ ]:
train_split = TrainingDataset(train_examples)
val_split   = TrainingDataset(val_examples)
test_split  = TrainingDataset(test_examples) # Isolated Gold set for final benchmark

logger.info(f"Grouped Stats: Train={len(train_split)} | Val={len(val_split)} | Test={len(test_split)}")

for ds_name, ds in {"train": train_split, "val": val_split, "test": test_split}.items():
    logger.info(f"Dataset: {ds_name} ")
    ds.print_stats()
    logger.debug(ds[0])

# Model Optimization & Fine-tuning

## Hyperparameter & Architecture Configuration

In [ ]:
experiment_name = f"gliner2_archaeo_lora_{datetime.now().strftime('%Y%m%d_%H%M')}"
output_dir = DATA_DIR / "models" / experiment_name
output_dir.mkdir(parents=True, exist_ok=True)

num_epochs = 20

training_config = TrainingConfig(
    output_dir=str(output_dir),
    experiment_name=experiment_name,
    seed=42,

    # Hardware & Batching Stability
    batch_size=1,
    eval_batch_size=1,             # Prevents "tensor size mismatch" during evaluation
    gradient_accumulation_steps=4, # Simulates Effective Batch Size = 4
    fp16=True,                     # Half-precision for speed/memory

    # LoRA Architecture (Rank 4 for stability on small datasets)
    use_lora=True,
    lora_r=4,                     # Reduced from 16
    lora_alpha=8.0,               # Reduced from 32.0 (standard 2*r)
    lora_dropout=0.1,             # Regularization for small datasets
    lora_target_modules=["encoder"], # Focused target
    save_adapter_only=True,        # Saves ~10-30MB instead of 1.2GB per checkpoint

    # Optimization Profile
    num_epochs=num_epochs,
    task_lr=1e-4,                 # Primary learning rate for adapters/heads
    warmup_ratio=0.1,
    scheduler_type="cosine",       # Smooth decay for stable convergence
    weight_decay=0.01,

    # Evaluation & Logging
    eval_strategy="epoch",         # Saves best checkpoint at end of every epoch
    save_best=True,
    report_to_wandb=wandb_enabled,
    wandb_project=env_vars.get("WANDB_PROJECT", "archaeo-ner-greek"),
    metric_for_best="f1",        # Use F1 to drive selection
    greater_is_better=True,      # Higher is better
    save_total_limit=2,
    logging_steps=5,

    # Early Stopping (DISABLED due to gliner2 v1.2.5 bug)
    early_stopping=False,
    early_stopping_patience=10,

    # Data Handling
    validate_data=True,
)

# Initialize WandB run & capture logging function
log_to_wandb = setup_wandb(wandb_enabled, training_config.wandb_project, experiment_name, training_config)

## Base Model Instantiation

In [ ]:
# Load the GLiNER2 base model
model = GLiNER2.from_pretrained("urchade/gliner2-small-v0.1")
logger.info(f"Base model loaded: urchade/gliner2-small-v0.1")

## Model Training

In [ ]:
# Initialize trainer
trainer = GLiNER2Trainer(
    model=model,
    config=training_config,
    train_dataset=train_split,
    eval_dataset=val_split,
)

# Run training
logger.info("Starting training...")
trainer.train()
logger.info(f"Training completed! Model saved to {output_dir}")

## Model Evaluation

In [ ]:
# Evaluate on test set
logger.info("Evaluating on test set...")
test_results = trainer.evaluate(test_split)
logger.info(f"Test Results: {test_results}")

# Save results
results_path = output_dir / "test_results.json"
with open(results_path, 'w') as f:
    json.dump(test_results, f, indent=2)
logger.info(f"Results saved to {results_path}")